<a href="https://colab.research.google.com/github/siregarmr/monetary-window-stress-test/blob/main/Monetary_Window_Stress_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================================================
# MONETARY WINDOW STRESS-TEST SIMULATION
#
# Notes:
# (1) This is a Monte Carlo simulation of systemic stress in the U.S. dollar
# monetary order, inspired by @postaperdavide's thought-provoking posts and
# comments on X, leading to article "The Closing Window: A Risk Analysis, Not a
# Prediction" at https://x.com/i/status/2045157814310453356.
# (2) The article described the CLosing Window and the risk of sudden,
# catastrophic failure. Such sudden risk of catastrophic failure is named "Slam"
# in this simulation, characterized as nonlinear, and quantified as probability
# distribution over time.
# (3) This is a simplified stress‑test model built for educational and discussion
# purposes only. It does not predict the future. It does not constitute financial
# or investment advice. Reflexivity is not modeled. The act of observing and
# publishing these results may alter the behavior of market participants,
# accelerating or decelerating the timeline.
# ===============================================================================

# -------------------------------------------------------------------------------
# Preamble
# -------------------------------------------------------------------------------
!pip install pandas-datareader yfinance --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import warnings
warnings.filterwarnings('ignore')

import pandas_datareader.data as web
import yfinance as yf

# -------------------------------------------------------------------------------
# Helper (to safely extract scalar)
# -------------------------------------------------------------------------------
def to_scalar(val):
    """Convert pandas Series or single-element array to float."""
    if isinstance(val, (pd.Series, pd.DataFrame)):
        return val.iloc[0] if len(val) > 0 else np.nan
    elif isinstance(val, np.ndarray):
        return val.item() if val.size == 1 else val[0]
    else:
        return val

# ===============================================================================
# 1. FETCH LIVE PUBLIC DATA
# ===============================================================================

print("Fetching live market data...")

# Set date range (last 2 years)
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=365*2)

# --- FRED Data ---
try:
    debt_gdp_series = web.DataReader('GFDEGDQ188S', 'fred', start_date, end_date)
    debt_gdp = to_scalar(debt_gdp_series.dropna().iloc[-1]) / 100
except Exception as e:
    print(f"FRED Debt/GDP fetch failed: {e}, using fallback.")
    debt_gdp = 1.278  # Fallback ~127.8%

try:
    fed_funds_series = web.DataReader('FEDFUNDS', 'fred', start_date, end_date)
    fed_funds = to_scalar(fed_funds_series.dropna().iloc[-1]) / 100
except:
    fed_funds = 0.0433  # Fallback April 2026

try:
    yield_10y_series = web.DataReader('DGS10', 'fred', start_date, end_date)
    yield_10y = to_scalar(yield_10y_series.dropna().iloc[-1]) / 100
except:
    yield_10y = 0.043  # Fallback

# --- VIX from Yahoo Finance ---
try:
    vix_data = yf.download('^VIX', period='5d', progress=False)
    if not vix_data.empty:
        vix_current = to_scalar(vix_data['Close'].iloc[-1])
    else:
        vix_current = 20.0
except:
    vix_current = 20.0

# --- IMF COFER Data (Manual, latest public release Q4 2025) ---
usd_reserve_share = 0.563    # 56.3%

# --- Foreign Holdings of U.S. Treasuries (TIC Data, Feb 2026 release) ---
foreign_holdings = 9.49e12   # $9.49 Trillion
total_debt = 38.98e12        # $38.98 Trillion
foreign_pct = foreign_holdings / total_debt

# --- Geopolitical Risk Index (FRED series GEPUCURRENT) ---
try:
    gpr_series = web.DataReader('GEPUCURRENT', 'fred', start_date, end_date)
    gpr = to_scalar(gpr_series.dropna().iloc[-1])
except:
    gpr = 180.0  # Fallback

print(f"Debt-to-GDP:        {debt_gdp:.2%}")
print(f"Fed Funds Rate:     {fed_funds:.2%}")
print(f"10Y Treasury Yield: {yield_10y:.2%}")
print(f"VIX:                {vix_current:.2f}")
print(f"USD Reserve Share:  {usd_reserve_share:.2%}")
print(f"Foreign Holding %:  {foreign_pct:.2%}")
print(f"Geopolitical Risk:  {gpr:.1f}")
print("\nData fetch complete.")

In [ ]:
# ===============================================================================
# 2. SIMULATION PARAMETERS & MODEL
# ===============================================================================

# -------------------------------------------------------------------------------
# User Adjustable Parameters
# -------------------------------------------------------------------------------
SIM_YEARS = 15               # Extended horizon (we analyze 5, 10, and 15 years)
N_SIMULATIONS = 10000        # Monte Carlo runs
TIME_STEPS = 252 * SIM_YEARS # Daily steps

# Initial values from live data (ensure scalar float)
INIT_DEBT_GDP = float(debt_gdp)
INIT_YIELD = float(yield_10y)
INIT_VIX = float(vix_current) / 100.0
INIT_USD_SHARE = float(usd_reserve_share)
INIT_FOREIGN_PCT = float(foreign_pct)
INIT_GPR = float(np.ravel(gpr)[0]) / 500.0

# -------------------------------------------------------------------------------
# Stress Function (Vectorized)
# -------------------------------------------------------------------------------
def system_stress(debt_gdp, yield_rate, vix, usd_share, foreign_pct, gpr):
    debt_stress = np.minimum(debt_gdp / 1.5, 1.0)
    yield_stress = np.minimum(yield_rate / 0.10, 1.0)
    vix_stress = np.minimum(vix / 0.40, 1.0)
    trust_factor = (usd_share * 0.6 + foreign_pct * 0.4)
    trust_stress = 1 - trust_factor
    gpr_stress = np.minimum(gpr / 1.0, 1.0)

    S = (0.25 * debt_stress +
         0.15 * yield_stress +
         0.10 * vix_stress +
         0.30 * trust_stress +
         0.20 * gpr_stress)
    return S

# -------------------------------------------------------------------------------
# Slam Threshold (Baseline Reference)
# -------------------------------------------------------------------------------
SLAM_THRESHOLD = 0.65    # Calibrated to historical stress events

# -------------------------------------------------------------------------------
# Compute Initial Stress for Normalization
# -------------------------------------------------------------------------------
INIT_STRESS = system_stress(INIT_DEBT_GDP, INIT_YIELD, INIT_VIX,
                            INIT_USD_SHARE, INIT_FOREIGN_PCT, INIT_GPR)
print(f"Initial Stress Score (raw): {INIT_STRESS:.4f}")

# Normalized threshold: buffer above current stress
SLAM_BUFFER = 0.12             # Buffer large enough to absorb 1-2 shocks
EFFECTIVE_THRESHOLD = SLAM_BUFFER
print(f"Effective Threshold (normalized): {EFFECTIVE_THRESHOLD:.4f}")

# -------------------------------------------------------------------------------
# Shock Parameters
# -------------------------------------------------------------------------------
# SHOCK_PROB_DAILY = 0.001     # Conservative; ~0.1% per day (~22% chance a year)
SHOCK_PROB_DAILY = 0.003       # Pessimistic; ~0.3% per day (~53% chance a year)
SHOCK_MAGNITUDE_YIELD = 0.015  # +1.5% instantaneous yield spike
SHOCK_MAGNITUDE_VIX = 0.15     # +15 VIX points
SHOCK_MAGNITUDE_USD = -0.02    # -2% USD reserve share drop

# -------------------------------------------------------------------------------
# Stochastic Processes with Shocks
# -------------------------------------------------------------------------------
def simulate_paths(n_steps, n_sims, init_vals):
    dt = 1.0 / 252.0
    n_vars = 6
    paths = np.zeros((n_steps, n_sims, n_vars))

    paths[0, :, 0] = init_vals['debt_gdp']
    paths[0, :, 1] = init_vals['yield']
    paths[0, :, 2] = init_vals['vix']
    paths[0, :, 3] = init_vals['usd_share']
    paths[0, :, 4] = init_vals['foreign_pct']
    paths[0, :, 5] = init_vals['gpr']

    # Drift. Based on CBO long-term projections. Net drift ~3% is conservative.
    mu_dg = 0.03 / 252.0

    # Volatility. 0.002 daily (~3.2% annually) is conservative.
    sigma_dg = 0.002

    # Long-term yield anchor.
    long_term_yield = 0.045
    mr_yield = 0.1
    sigma_yield = 0.005

    # Long-term VIX anchor.
    long_term_vix = 0.20
    mr_vix = 0.2
    sigma_vix = 0.015

    # Drift in USD reserve share. 1% annual decline.
    drift_usd = -0.01 / 252.0
    sigma_usd = 0.003

    # Drift in foreign holdings percentage. 0.5% annual decline.
    drift_fp = -0.005 / 252.0
    sigma_fp = 0.002

    # Long-term Geopolitical Risk anchor.
    long_term_gpr = 0.25
    mr_gpr = 0.05
    sigma_gpr = 0.01

    for t in range(1, n_steps):
        shock_mask = np.random.random(n_sims) < SHOCK_PROB_DAILY

        # Debt/GDP
        dW_dg = np.random.normal(0, np.sqrt(dt), n_sims)
        paths[t,:,0] = (paths[t-1,:,0] + mu_dg * dt + sigma_dg * dW_dg)
        paths[t,:,0] = np.clip(paths[t,:,0], 0.8, 2.5)

        # Yield
        dW_y = np.random.normal(0, np.sqrt(dt), n_sims)
        paths[t,:,1] = (paths[t-1,:,1]
                        + mr_yield * (long_term_yield - paths[t-1,:,1]) * dt
                        + sigma_yield * dW_y)
        paths[t,:,1] += shock_mask * SHOCK_MAGNITUDE_YIELD
        paths[t,:,1] = np.clip(paths[t,:,1], 0.0, 0.15)

        # VIX
        dW_v = np.random.normal(0, np.sqrt(dt), n_sims)
        paths[t,:,2] = (paths[t-1,:,2]
                        + mr_vix * (long_term_vix - paths[t-1,:,2]) * dt
                        + sigma_vix * dW_v)
        paths[t,:,2] += shock_mask * SHOCK_MAGNITUDE_VIX
        paths[t,:,2] = np.clip(paths[t,:,2], 0.10, 0.80)

        # USD Share
        dW_usd = np.random.normal(0, np.sqrt(dt), n_sims)
        paths[t,:,3] = (paths[t-1,:,3] + drift_usd * dt + sigma_usd * dW_usd)
        paths[t,:,3] += shock_mask * SHOCK_MAGNITUDE_USD
        paths[t,:,3] = np.clip(paths[t,:,3], 0.30, 0.70)

        # Foreign holdings
        dW_fp = np.random.normal(0, np.sqrt(dt), n_sims)
        paths[t,:,4] = (paths[t-1,:,4] + drift_fp * dt + sigma_fp * dW_fp)
        paths[t,:,4] = np.clip(paths[t,:,4], 0.10, 0.50)

        # Geopolitical risk
        dW_g = np.random.normal(0, np.sqrt(dt), n_sims)
        paths[t,:,5] = (paths[t-1,:,5]
                        + mr_gpr * (long_term_gpr - paths[t-1,:,5]) * dt
                        + sigma_gpr * dW_g)
        paths[t,:,5] = np.clip(paths[t,:,5], 0.0, 1.0)

    return paths

print("Simulation engine ready (with shocks, horizon=15y).")

In [ ]:
# ===============================================================================
# 3. RUN SIMULATION AND COMPUTE PROBABILITIES (WITH NORMALIZATION)
# ===============================================================================

init_vals = {
    'debt_gdp': INIT_DEBT_GDP,
    'yield': INIT_YIELD,
    'vix': INIT_VIX,
    'usd_share': INIT_USD_SHARE,
    'foreign_pct': INIT_FOREIGN_PCT,
    'gpr': INIT_GPR
}

print(f"Running {N_SIMULATIONS} scenarios over {SIM_YEARS} years...")
paths = simulate_paths(TIME_STEPS, N_SIMULATIONS, init_vals)

# Compute raw stress scores
stress_scores_raw = np.zeros((TIME_STEPS, N_SIMULATIONS))
for t in range(TIME_STEPS):
    debt = paths[t, :, 0]
    yld  = paths[t, :, 1]
    vix  = paths[t, :, 2]
    usd  = paths[t, :, 3]
    fp   = paths[t, :, 4]
    gpr  = paths[t, :, 5]
    stress_scores_raw[t, :] = system_stress(debt, yld, vix, usd, fp, gpr)

# Normalize by subtracting initial stress
stress_scores = stress_scores_raw - INIT_STRESS

# Find slam times using EFFECTIVE_THRESHOLD
slam_times = np.full(N_SIMULATIONS, np.inf)
for sim in range(N_SIMULATIONS):
    cross_idx = np.where(stress_scores[:, sim] > EFFECTIVE_THRESHOLD)[0]
    if len(cross_idx) > 0:
        slam_times[sim] = cross_idx[0]

slam_years = slam_times / 252

# --- Analyze multiple horizons ---
horizons = [5, 10, 15]
print(f"\n=== SIM RESULTS (Effective Threshold = {EFFECTIVE_THRESHOLD:.2f}) ===")
for h in horizons:
    days_h = h * 252
    prob = np.mean(slam_times <= days_h)
    print(f"Probability of Slam within {h:2d} years: {prob:.1%}")
    if prob > 0:
        cond_years = slam_years[slam_years <= h]
        median_y = np.median(cond_years)
        print(f"  -> Median time to Slam (within horizon): {median_y:.2f} years")

In [ ]:
# ===============================================================================
# 4. VISUALIZATION (FIXED MEDIAN REFERENCE)
# ===============================================================================

import matplotlib.pyplot as plt
%matplotlib inline
plt.ion()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Subplot 1: Stress Score Evolution ---
ax1 = axes[0,0]
time_years = np.arange(TIME_STEPS) / 252
p10 = np.percentile(stress_scores, 10, axis=1)
p50 = np.percentile(stress_scores, 50, axis=1)
p90 = np.percentile(stress_scores, 90, axis=1)
ax1.fill_between(time_years, p10, p90, alpha=0.3, color='orange')
ax1.plot(time_years, p50, 'b-', linewidth=2, label='Median Stress')
ax1.axhline(SLAM_THRESHOLD, color='red', linestyle='--',
            label=f'Slam Threshold ({SLAM_THRESHOLD})')
ax1.set_xlabel('Years from Today')
ax1.set_ylabel('System Stress Score (S)')
ax1.set_title('Monetary Window Stress Projection')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Subplot 2: Histogram of Slam Times ---
ax2 = axes[0,1]
slam_years_occ = slam_years[slam_years < np.inf]
if len(slam_years_occ) > 0:
    ax2.hist(slam_years_occ, bins=30, color='darkred', alpha=0.7,
             edgecolor='black')
    ax2.axvline(np.median(slam_years_occ), color='blue', linestyle='--',
                label=f'Median: {np.median(slam_years_occ):.2f} yrs')
    ax2.set_xlabel('Years until Slam')
    ax2.set_ylabel('Frequency')
    ax2.set_title(f'Slam Timing Distribution (N={len(slam_years_occ)})')
    ax2.legend()
else:
    ax2.text(0.5, 0.5, 'No Slam Events Simulated', ha='center',
             va='center', transform=ax2.transAxes)
ax2.grid(True, alpha=0.3)

# --- Subplot 3: Sample Yield Paths ---
ax3 = axes[1,0]
sample_sims = np.random.choice(N_SIMULATIONS, 5, replace=False)
for sim in sample_sims:
    ax3.plot(time_years, paths[:, sim, 1]*100, alpha=0.7)
ax3.set_xlabel('Years')
ax3.set_ylabel('10-Year Treasury Yield (%)')
ax3.set_title('Sample Yield Paths (5 Random Simulations)')
ax3.grid(True, alpha=0.3)

# --- Subplot 4: USD Reserve Share ---
ax4 = axes[1,1]
p10_usd = np.percentile(paths[:, :, 3]*100, 10, axis=1)
p50_usd = np.percentile(paths[:, :, 3]*100, 50, axis=1)
p90_usd = np.percentile(paths[:, :, 3]*100, 90, axis=1)
ax4.fill_between(time_years, p10_usd, p90_usd, alpha=0.3, color='green')
ax4.plot(time_years, p50_usd, 'g-', linewidth=2, label='Median USD Share')
ax4.set_xlabel('Years')
ax4.set_ylabel('USD Share of Global Reserves (%)')
ax4.set_title('Projected Erosion of Dollar Hegemony')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Fallback save
fig.savefig('monetary_window_simulation.png', dpi=150, bbox_inches='tight')
from IPython.display import Image
Image('monetary_window_simulation.png')

print("\n=== ANNUAL SLAM PROBABILITY ===")
for year in range(1, min(SIM_YEARS, 15)+1):
    days = year * 252
    prob_year = np.mean((slam_times <= days) & (slam_times > (year-1)*252))
    print(f"Year {year:2d}: {prob_year:.2%} chance of slam occurring")